# Budget-Aware AI Evaluation: Cost-Optimal Sampling for LLM Judges

When evaluating an AI system, you often have access to two annotation sources: a cheap but biased LLM-as-judge and an expensive but accurate human annotator. Given a fixed budget, how should you split it between them?

**Cost-Optimal Random Sampling** computes the fraction of samples to send to the expensive annotator so that estimation variance is minimised within your budget.

---

**What you will learn:**

- How to use `CostOptimalRandomSampler` to compute the optimal annotation fraction from a small labeled dataset
- What the `pi` and `xi` outputs represent and how they feed into inverse probability weighting
- How to combine them with `ASIMeanEstimator` for a valid, bias-corrected estimate
- How cost-optimal sampling outperforms both a proxy-only and a human-only strategy across a range of budgets

## The evaluation challenge: two annotation sources, one budget

Suppose you are measuring the hallucination rate of a customer-facing AI assistant. Two annotation sources are available:

- **LLM-as-judge**: $0.05 per sample. Fast and scalable, but systematically under-reports hallucinations.
- **Human annotator**: $10.00 per sample. Ground-truth quality, but 200 times more expensive.

### What you already have

Before the main evaluation, you ran a small pilot study and annotated **100 conversations** with **both** sources. This burn-in dataset serves two purposes:

1. It reveals that the judge is biased: the judge reports a hallucination rate of ~5%, while human annotators find ~10%.
2. It allows you to **calibrate** the cost-optimal sampler before spending the main budget.

### The question

You now have **2,000 new conversations** to evaluate and a budget of **$1,500**. Each dollar must cover both the proxy cost ($0.05/sample) and any human annotation cost ($10.00/sample). How do you allocate this budget?

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ASIMeanEstimator, ClassicalMeanEstimator, PPIMeanEstimator
from glide.samplers import CostOptimalRandomSampler
from glide.simulators import generate_binary_dataset

N_BURN_IN = 100
N_MAIN = 2000
N_TOTAL = N_BURN_IN + N_MAIN
TRUE_RATE = 0.10
PROXY_RATE = 0.05
COST_PROXY = 0.05
COST_HUMAN = 10.00
BUDGET_DEMO = 1500
RANDOM_SEED = 0

C_OPTIMAL = "#27AE60"  # cost-optimal — green
C_PROXY = "#E67E22"  # proxy only   — orange
C_HUMAN = "#2E86AB"  # human only   — blue
C_TRUTH = "#2C3E50"  # true value   — dark slate

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titlepad": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    }
)

## Simulating 2,100 conversations with a biased judge

`generate_binary_dataset_with_oracle_sampling` produces a synthetic dataset that matches the scenario above.

| Array | Meaning |
|---|---|
| `y_true_oracle` | Ground-truth binary label: $Y_i = 1$ means a hallucination confirmed by a human annotator |
| `y_proxy` | Binary label from the LLM judge: $\tilde{Y}_i = 1$ means the judge flagged a hallucination |

The first `N_BURN_IN = 100` conversations are annotated with **both** sources: these form the burn-in dataset used to calibrate the sampler. The remaining 2,000 conversations are new, and only the proxy label is available initially.

> **Simulated annotation:** In practice, a human label for a new conversation is revealed only after a human annotator is paid. Here we generate all ground-truth labels upfront to simulate the annotation process: a main-set conversation only gets its true label revealed when the sampler decides to annotate it (i.e. when $\xi_i = 1$).

In [ ]:
y_true, y_proxy = generate_binary_dataset(
    n_labeled=N_TOTAL,
    n_unlabeled=0,
    true_mean=TRUE_RATE,
    proxy_mean=PROXY_RATE,
    correlation=0.55,
    random_seed=RANDOM_SEED,
)

y_true_burn_in = y_true[:N_BURN_IN]
y_proxy_burn_in = y_proxy[:N_BURN_IN]

In [ ]:
print(f"Total conversations   : {N_TOTAL:,}")
print(f"  Burn-in (labeled)   : {N_BURN_IN}")
print(f"  Main (new)          : {N_MAIN:,}")
print()
print(f"LLM judge cost        : ${COST_PROXY:.2f} per sample")
print(f"Human annotator cost  : ${COST_HUMAN:.2f} per sample")
print(f"Cost ratio            : {COST_HUMAN / COST_PROXY:.0f}x")
print()
print(f"Burn-in: judge rate   = {y_proxy_burn_in.mean():.1%}")
print(f"Burn-in: human rate   = {y_true_burn_in.mean():.1%}")
print(f"Proxy bias            = {y_proxy_burn_in.mean() - y_true_burn_in.mean():+.1%}")

## CostOptimalRandomSampler to the rescue

The burn-in data confirms the proxy is biased, so we cannot trust it directly. We need prediction-powered estimation to correct for this bias. The question is: how many new human annotations should we purchase alongside the cheap proxy labels?

`CostOptimalRandomSampler` answers exactly this question. The workflow has two steps:

1. **Fit** the sampler on the burn-in dataset to estimate the proxy's reliability.
2. **Sample** from the new conversations to obtain annotation assignments.

### Step 1: Fit on the burn-in dataset

The sampler estimates two quantities from the burn-in data:

- $\text{Var}(H)$: variance of the ground-truth labels, capturing how spread out the true outcomes are.
- $\text{MSE}(H, G)$: mean squared error of the proxy relative to ground truth, measuring how unreliable the judge is.

Both quantities are computed during `sampler.fit(y_true, y_proxy)` using the burn-in samples.

In [ ]:
sampler = CostOptimalRandomSampler()
sampler.fit(y_true_burn_in, y_proxy_burn_in)

print(f"Estimated Var(H)    : {sampler._y_true_variance:.4f}")
print(f"Estimated MSE(H, G) : {sampler._mean_squared_error:.4f}")

### The optimal annotation rate

With $\text{Var}(H)$ and $\text{MSE}(H, G)$ estimated from burn-in, the sampler derives $\pi^{*}$, the probability at which each new conversation is sent for human annotation:

$$\pi^{*} = \sqrt{\frac{c_g}{c_h} \cdot \frac{\text{MSE}(H, G)}{\text{Var}(H) - \text{MSE}(H, G)}}$$

Two intuitions from this formula:

- A **higher cost ratio** $c_h / c_g$ drives $\pi^{*}$ down: when human annotation is very expensive relative to the proxy, fewer human annotations are needed to stay within budget.
- A **more accurate proxy** (small $\text{MSE}$) also drives $\pi^{*}$ down: when the judge closely tracks the truth, only a small fraction of human annotations is needed to correct the residual bias.

The expected cost per new conversation is $c_g + c_h \cdot \pi^{*}$. Given budget $B$ and $N$ new conversations, the sampler can process at most $T = \lfloor B / (c_g + c_h \cdot \pi^{*}) \rfloor$ of them.

In [ ]:
pi_star = sampler._compute_optimal_probability(COST_HUMAN, COST_PROXY)
cost_per_sample_optimal = COST_PROXY + COST_HUMAN * pi_star
T_max = int(np.floor(BUDGET_DEMO / cost_per_sample_optimal))

cost_per_sample_human_only = COST_HUMAN
n_human_only_naive = int(np.floor(BUDGET_DEMO / cost_per_sample_human_only))

cost_per_sample_proxy_only = COST_PROXY
n_proxy_only_naive = int(np.floor(BUDGET_DEMO / cost_per_sample_proxy_only))

print(f"Optimal annotation rate        : pi* = {pi_star:.3f}")
print(f"Expected cost per sample       : ${cost_per_sample_optimal:.3f}")
print(f"Conversations covered (budget) : T = {T_max}  (all {N_MAIN:,} fit: {T_max >= N_MAIN})")
print()
print("Compare: at pi=1 (full annotation each time),")
print(f"same budget covers only {n_human_only_naive} conversations")

### Step 2: Draw annotation assignments for new conversations

With $\pi^{*}$ determined, the sampler draws independent Bernoulli indicators $\xi_i \sim \text{Bernoulli}(\pi^{*})$ for each new conversation. Conversations with $\xi_i = 1$ are sent for human annotation alongside the LLM label. Conversations with $\xi_i = 0$ receive only the LLM label.

`sampler.sample()` returns three values:

| Return value | Meaning |
|---|---|
| `indices` | Indices of the conversations included in this run (all N\_MAIN when budget is sufficient) |
| `pi` | The annotation probability $\pi^{*}$ shared by all new conversations |
| `xi` | Bernoulli indicators: $1$ = human annotation obtained, $0$ = proxy label only |

In [ ]:
indices, pi, xi = sampler.sample(
    n_samples=N_MAIN,
    y_true_cost=COST_HUMAN,
    y_proxy_cost=COST_PROXY,
    budget=BUDGET_DEMO,
    random_seed=RANDOM_SEED,
)

n_human_main = int(xi.sum())

print(f"Conversations processed   : {len(indices):,}  (of {N_MAIN:,})")
print(f"Human annotations drawn   : {n_human_main}  ({n_human_main / len(indices):.1%} of processed)")
print(f"Annotation probability    : pi = {pi:.3f}")
print(f"Realized annotation rate  : {n_human_main / len(indices):.3f}")

## Understanding `pi` and `xi`

The two outputs encode the **sampling design**: they tell downstream estimators exactly how the labeled subset was drawn.

- **`xi`** (sampling indicator): a binary vector of length `N_MAIN`. `xi[i] = 1` means conversation `i` was sent for human annotation. `xi[i] = 0` means only its proxy label was obtained.

- **`pi`** (annotation probability): the probability that any given conversation had `xi = 1`. Here all conversations share the same value $\pi = \pi^{*}$, since cost-optimal random sampling applies a uniform Bernoulli probability across all new conversations.

The ratio $1/\pi$ is used by the estimator to reweight labeled samples. A conversation annotated with probability $\pi^{*} = 0.2$ represents, on average, 5 such conversations in the population. Dividing its contribution by $\pi^{*}$ corrects for this underrepresentation, a technique called **inverse probability weighting (IPW)**.

In [ ]:
print(f"xi values (first 10 main conversations): {xi[:10]}")
print(f"pi value (same for all)                : {pi:.4f}")
print()
print(f"Proportion with xi=1 : {xi.mean():.3f}  (expected pi* = {pi:.3f})")
print()
print(f"Human annotation cost : {n_human_main} x ${COST_HUMAN:.2f} = ${n_human_main * COST_HUMAN:.2f}")
print(f"Proxy label cost      : {len(indices)} x ${COST_PROXY:.2f} = ${len(indices) * COST_PROXY:.2f}")
print(f"Total spent           : ${n_human_main * COST_HUMAN + len(indices) * COST_PROXY:.2f}  (budget: ${BUDGET_DEMO})")

## Combining `pi` and `xi` with `ASIMeanEstimator`

The `ASIMeanEstimator` implements **Active Statistical Inference (ASI)**, which extends prediction-powered inference to non-uniform sampling designs. It requires three inputs for the full dataset:

- `y_true`: ground-truth labels, with `np.nan` for conversations not sent for human annotation.
- `y_proxy`: proxy labels for every conversation.
- `pi`: annotation probability for every conversation.

For each sample, ASI computes an **IPW-corrected effective label**:

$$z_i(\lambda) = \lambda\, \tilde{Y}_i + \xi_i \cdot \frac{Y_i - \lambda\, \tilde{Y}_i}{\pi_i}$$

For labeled samples ($\xi_i = 1$), the residual $Y_i - \lambda\, \tilde{Y}_i$ is divided by $\pi_i$, upweighting contributions from samples that were less likely to be selected. For unlabeled samples ($\xi_i = 0$), the expression reduces to $\lambda\, \tilde{Y}_i$, so the proxy label still contributes. The parameter $\lambda$ (tuned automatically) controls the weight placed on the proxy.

The final estimate is the mean of these corrected labels: $\hat{\theta} = \frac{1}{n} \sum_i z_i(\lambda)$.

## Including the burn-in data in the final estimate

The burn-in conversations were annotated with certainty ($\pi = 1$ for all of them), so each of their contributions needs no IPW upweighting. They can be combined directly with the main-set results by setting their `pi` values to 1 in the full `pi` array.

This means the burn-in dataset does double duty:

1. It **calibrates** the sampler (used to compute $\pi^{*}$).
2. It **contributes labeled samples** to the final estimate (included in the full dataset passed to `ASIMeanEstimator`).

The estimator receives a `pi` array of length `N_TOTAL = N_BURN_IN + N_MAIN`, with `pi[:N_BURN_IN] = 1.0` for the burn-in and `pi[N_BURN_IN:] = pi*` for all main conversations.

## Running the full estimation pipeline

We now assemble the complete input for `ASIMeanEstimator`:

1. Build `y_true_full`: a length-`N_TOTAL` array where burn-in rows carry their ground-truth labels and main rows carry the ground-truth label only when `xi[i] = 1` (human annotation obtained), else `np.nan`.
2. Build `pi_full`: a length-`N_TOTAL` array with `1.0` for burn-in and `pi*` for all main conversations.
3. Pass `(y_true_full, y_proxy, pi_full)` to `ASIMeanEstimator().estimate()`.

In [ ]:
y_true_full = np.full(N_TOTAL, np.nan)
y_true_full[:N_BURN_IN] = y_true_burn_in

annotated_main_indices = indices[xi == 1]
y_true_full[N_BURN_IN + annotated_main_indices] = y_true_oracle[N_BURN_IN + annotated_main_indices]

pi_full = np.full(N_TOTAL, pi)
pi_full[:N_BURN_IN] = 1.0

result_demo = ASIMeanEstimator().estimate(
    y_true_full,
    y_proxy,
    pi_full,
    metric_name="Hallucination Rate",
    confidence_level=0.95,
)

In [ ]:
n_human_total_demo = N_BURN_IN + n_human_main
print(result_demo.summary())
print()
print(f"Human labels used : {n_human_total_demo}  ({N_BURN_IN} burn-in + {n_human_main} main)")
print(f"Proxy labels used : {N_TOTAL}")
print(f"True rate         : {TRUE_RATE:.1%}")

## How budget size affects confidence interval width

With cost-optimal sampling, spending more budget means processing more new conversations and purchasing more human annotations at rate $\pi^{*}$, reducing estimation variance. The plot below runs the full pipeline across a range of budgets and shows how the 95% confidence interval width shrinks as more budget is invested.

At very small budgets, only a fraction of the 2,000 new conversations is processed (T < N\_MAIN), so the estimate is dominated by the burn-in labeled set. As budget grows, more conversations enter the estimation and the CI narrows. Beyond the threshold where all N\_MAIN conversations are covered, additional budget no longer helps (the CI stabilises).

In [ ]:
BUDGETS = [100, 300, 700, 1500, 3000, 6000]

ci_widths_optimal = []
n_human_per_budget = []

for budget in BUDGETS:
    s = CostOptimalRandomSampler()
    s.fit(y_true_burn_in, y_proxy_burn_in)
    idx_b, pi_b, xi_b = s.sample(
        n_samples=N_MAIN,
        y_true_cost=COST_HUMAN,
        y_proxy_cost=COST_PROXY,
        budget=budget,
        random_seed=RANDOM_SEED,
    )

    y_t = np.full(N_TOTAL, np.nan)
    y_t[:N_BURN_IN] = y_true_burn_in
    ann_idx = idx_b[xi_b == 1]
    y_t[N_BURN_IN + ann_idx] = y_true_oracle[N_BURN_IN + ann_idx]

    pi_arr = np.full(N_TOTAL, pi_b)
    pi_arr[:N_BURN_IN] = 1.0

    r = ASIMeanEstimator().estimate(y_t, y_proxy, pi_arr)
    ci_widths_optimal.append(r.confidence_interval.upper_bound - r.confidence_interval.lower_bound)
    n_human_per_budget.append(N_BURN_IN + int(xi_b.sum()))

for budget, width, n_h in zip(BUDGETS, ci_widths_optimal, n_human_per_budget):
    print(f"  Budget ${budget:>5}: CI width = {width:.3f}  ({n_h} human labels)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(
    BUDGETS,
    ci_widths_optimal,
    color=C_OPTIMAL,
    linewidth=2.5,
    marker="o",
    markersize=7,
    label="Cost-optimal (ASI)",
)

ax.set_xlabel("Annotation Budget ($)", fontsize=12)
ax.set_ylabel("95% CI Width", fontsize=12)
ax.set_title("Higher Budget Reduces Estimation Uncertainty", fontsize=14, fontweight="bold")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Comparing with two baselines at a fixed budget

To understand where the gain comes from, we compare cost-optimal sampling against two strategies that each make one of the two possible mistakes at the same budget of $1,500:

**Proxy-only baseline:** spend the entire budget on proxy labels only, purchasing no new human annotations beyond the burn-in set. Since each proxy costs $0.05, all 2,000 new conversations can be proxy-labeled for just $100, well within budget. A PPI estimator is run with only the 100 burn-in samples as labeled data. The CI width is limited by those 100 human labels and does not improve with additional proxy spending.

**Human-only baseline:** spend the entire budget on full annotations, acquiring as many human-labeled conversations as possible ($10.05 each: human + proxy). No attempt is made to leverage proxy labels on the remaining conversations. A classical mean estimator is run on burn-in plus the purchased main annotations.

Cost-optimal sampling sits between these two extremes: it buys a meaningful number of human annotations to correct the proxy bias, and leverages all proxy labels for variance reduction via ASI.

In [ ]:
# Proxy-only: PPIMeanEstimator with burn-in as labeled, all main as unlabeled proxy
y_true_proxy_only = np.full(N_TOTAL, np.nan)
y_true_proxy_only[:N_BURN_IN] = y_true_burn_in
result_proxy_only = PPIMeanEstimator().estimate(
    y_true_proxy_only,
    y_proxy,
    metric_name="Hallucination Rate",
)

# Human-only: ClassicalMeanEstimator on burn-in + as many main samples as budget allows
n_human_only_main = int(np.floor(BUDGET_DEMO / (COST_PROXY + COST_HUMAN)))
rng = np.random.default_rng(RANDOM_SEED + 1)
human_only_main_idx = np.sort(rng.choice(N_MAIN, size=n_human_only_main, replace=False))
y_true_human_only = np.hstack(
    [
        y_true_burn_in,
        y_true_oracle[N_BURN_IN + human_only_main_idx],
    ]
)
result_human_only = ClassicalMeanEstimator().estimate(
    y_true_human_only,
    metric_name="Hallucination Rate",
)

n_human_total_human_only = N_BURN_IN + n_human_only_main
print(f"Proxy-only  : {N_BURN_IN} human labels + {N_TOTAL} proxy labels")
print(f"Human-only  : {n_human_total_human_only} human labels total  ({n_human_only_main} new at ${BUDGET_DEMO})")
print(f"Cost-optimal: {n_human_total_demo} human labels + {N_TOTAL} proxy labels  (pi* = {pi:.2f})")

In [ ]:
baseline_estimates = [
    (
        f"Proxy-only\n({N_BURN_IN} human  +  {N_TOTAL:,} proxy)\n(no new human annotations)",
        result_proxy_only.mean,
        result_proxy_only.confidence_interval.lower_bound,
        result_proxy_only.confidence_interval.upper_bound,
        C_PROXY,
    ),
    (
        f"Human-only\n({n_human_total_human_only} human  |  no PPI)\n(${BUDGET_DEMO} budget)",
        result_human_only.mean,
        result_human_only.confidence_interval.lower_bound,
        result_human_only.confidence_interval.upper_bound,
        C_HUMAN,
    ),
    (
        f"Cost-optimal (GLIDE)\n({n_human_total_demo} human  +  {N_TOTAL:,} proxy)\n"
        f"(pi* = {pi:.2f}, ${BUDGET_DEMO} budget)",
        result_demo.mean,
        result_demo.confidence_interval.lower_bound,
        result_demo.confidence_interval.upper_bound,
        C_OPTIMAL,
    ),
]

fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = [2, 1, 0]

for y, (label, mean, lo, hi, color) in zip(y_pos, baseline_estimates):
    ax.plot([lo, hi], [y, y], color=color, linewidth=4, solid_capstyle="round", zorder=3)
    for xc in [lo, hi]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2], color=color, linewidth=2.5, zorder=3)
    ax.scatter(mean, y, s=200, color=color, zorder=5, edgecolors="white", linewidths=2.5)
    ax.text(mean, y + 0.34, f"{mean:.1%}", ha="center", va="bottom", fontsize=12, color=color, fontweight="bold")
    ax.text(mean, y - 0.34, f"[{lo:.1%},  {hi:.1%}]", ha="center", va="top", fontsize=11, color="#888888")

ax.axvline(TRUE_RATE, color=C_TRUTH, linestyle="--", linewidth=2.5, zorder=4)
ax.text(TRUE_RATE + 0.004, 2.72, "True rate  10%", color=C_TRUTH, fontsize=10.5, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in baseline_estimates], fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Estimated Hallucination Rate", fontsize=12)
ax.set_title(
    "Cost-Optimal Sampling Outperforms Both Baselines",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlim(-0.01, 0.26)
ax.set_ylim(-0.8, 3.2)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Summary: what each strategy contributes

| | Proxy-only | Human-only | **Cost-optimal** |
|---|---|---|---|
| New proxy labels | 2,000 | 149 | 2,000 |
| New human labels | 0 | 149 | ~146 |
| Annotation probability | n/a | 1 | ~0.2 (optimal) |
| Unbiased estimate | limited by burn-in | yes | yes |
| Leverages all proxy labels | yes (via PPI) | no | yes (via ASI + IPW) |
| CI narrows with more budget | no | yes (slowly) | yes (efficiently) |

**Key takeaways:**

1. **The burn-in dataset does double duty.** It calibrates the sampler and contributes labeled samples to the final estimate. No data is wasted.

2. **Pi\* is determined by the cost ratio and proxy quality.** A 200x cost gap and a moderately accurate proxy yield pi\* around 0.2: only one in five new conversations needs a human label to achieve near-optimal precision.

3. **Proxy-only is limited by burn-in.** Spending your remaining budget entirely on cheap proxy labels gives you many more samples but adds no new ground-truth signal beyond what the burn-in already provides. The CI does not shrink.

4. **Human-only wastes the proxy labels.** Spending your budget on full annotations ignores the variance reduction available from leveraging all proxy labels. You get fewer human labels per dollar and no PPI benefit.

5. **Cost-optimal gets the best of both worlds.** It covers all new conversations with proxy labels and spends just enough on human annotations to correct the bias. The same budget yields tighter confidence intervals than either alternative.

---

*Want to go further? The [ASI tutorial](../asi/) shows how to concentrate human annotations on the most uncertain conversations, adding an active layer on top of cost optimisation.*